# Train the Cross-Modal Satellite Retrieval Model (Colab T4)

This notebook runs the full training pipeline on a **free Colab T4 GPU** and exports the model to ONNX for CPU inference on the laptop.

**Runtime → Change runtime type → T4 GPU** before running.

Steps:
1. Clone the repo + install deps.
2. (Optional) mount Google Drive for checkpoint persistence.
3. Prepare the SEN12MS subset (real if reachable, else synthetic — still learnable).
4. Train with InfoNCE contrastive loss.
5. Export three ONNX files (one per modality).

## 1. Setup

In [ ]:
# Clone or upload the project. Replace URL with your repo.
!git clone https://github.com/<you>/cross-modal-satellite-retrieval.git /content/repo 2>/dev/null || echo 'repo already present'
%cd /content/repo
!pip install -q -r requirements.txt

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('/content/repo').resolve()))

import torch
print('GPU available:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## 2. (Optional) Mount Google Drive for checkpoints

Colab disconnects after ~12 h on the free tier. Pointing the checkpoint dir at Drive means a disconnect only costs at most one epoch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Override checkpoint dir to live on Drive.
import os
os.environ['CKPT_DIR'] = '/content/drive/MyDrive/sat_retrieval_ckpts'

## 3. Prepare the dataset

In [ ]:
# Try the real SEN12MS mirrors first; if unreachable, synthesise a learnable
# subset so the pipeline still runs end-to-end.
!python data/download_sen12ms.py --num 8000 --out data/sen12ms --size 64

## 4. Train (InfoNCE, shared ViT-S)

Move the checkpoint dir to Drive if mounted, so progress survives disconnects.

In [ ]:
from config import load_config
cfg = load_config()
import os
if os.environ.get('CKPT_DIR'):
    cfg['paths']['checkpoint_dir'] = os.environ['CKPT_DIR']
    import pathlib; pathlib.Path(cfg['paths']['checkpoint_dir']).mkdir(parents=True, exist_ok=True)
# Smaller subset => fewer epochs still converge; config default is 30.
cfg['train']['epochs'] = 30
cfg['train']['batch_size'] = 256
cfg

In [ ]:
from training import run_training
best_ckpt = run_training(cfg)
print('BEST:', best_ckpt)

## 5. Export to ONNX (one graph per modality)

In [ ]:
from training.export_onnx import export_all
onnx_files = export_all(cfg, best_ckpt)
onnx_files

## 6. (Optional) Quick CPU sanity check

Confirm the ONNX model produces L2-normalised 512-d vectors.

In [ ]:
import onnxruntime as ort, numpy as np
sess = ort.InferenceSession(str(onnx_files[1]), providers=['CPUExecutionProvider'])
x = np.random.randn(4, 3, 64, 64).astype(np.float32)
z = sess.run(None, {'image': x})[0]
print('embedding shape:', z.shape, '| row norms:', np.linalg.norm(z, axis=1))

## 7. Download the artifacts to your laptop

Move `models/retrieval_model_*.onnx` and the gallery index to the laptop to run the FastAPI UI locally.

In [ ]:
from google.colab import files
for f in onnx_files:
    files.download(str(f))